## Dependency note

This notebook needs [UCLCHEM](https://github.com/uclchem/UCLCHEM), which is a
Fortran-backed astrochemistry code with no PyPI wheel. It is **not** installed by
`pip install pycalima`, and it cannot be declared as a pyCALIMA extra because
PEP 508 direct references are rejected in distribution metadata.

Build it from source following the upstream instructions, then run this notebook
in the same environment. See `requirements-dev.txt` for the pinned reference.



# UCLCHEM notebook scaffold: multi-ice mantles + parallel grid runs

This notebook is designed to help you **build the workflow yourself** for deriving a mantle-induced accretion-efficiency reduction from UCLCHEM.

The focus is now on:
- identifying the **dominant ice species** covering grains,
- building a **composite ice-cover metric** rather than a CO-only metric,
- running a **large parameter grid in parallel** with Python multiprocessing,
- exporting either a lookup table or a fitted analytic blocking function for RAMSES.

UCLCHEM treats chemistry in the gas phase and in both the **surface and bulk of the ice mantles**, so this notebook is built around extracting those ice-reservoir outputs from each run. [web:44][page:1]


Some references to keep in mind:
https://arxiv.org/abs/2102.09862
https://ui.adsabs.harvard.edu/abs/2020PhRvL.124v1103P/abstract
https://arxiv.org/pdf/2005.00757
https://ui.adsabs.harvard.edu/abs/2021MNRAS.507.6205S/abstract
https://ui.adsabs.harvard.edu/abs/2021A%26A...648A..84M/abstract
https://www.aanda.org/articles/aa/full_html/2025/08/aa54257-25/aa54257-25.html
https://arxiv.org/abs/1310.8466
https://www.aanda.org/articles/aa/full_html/2020/11/aa39092-20/aa39092-20.html



## 0 · Strategy

The notebook is organized in this order:

1. Verify UCLCHEM and inspect one run.
2. Identify which ice species dominate the mantle under your conditions.
3. Define a parameter grid over \(n_{
m H}, T, G_0\), grain size, and optionally grain type.
4. Run the grid in parallel.
5. Build a **covering metric** from the dominant ice species.
6. Convert that covering metric into an accretion-efficiency reduction.
7. Fit a compact function or export a lookup table.

The reason for starting with diagnostics is that species naming can differ by network, and UCLCHEM outputs both surface and mantle phases depending on the setup. [web:44]


In [ ]:

import os
import tempfile
from pathlib import Path
from itertools import product
from multiprocessing import get_context, cpu_count

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import uclchem

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25



## 1 · One diagnostic run

Do one run at representative conditions from your simulation. The purpose is to inspect the output structure and naming conventions before scaling up. The UCLCHEM site emphasizes that the code treats both surface and bulk mantle chemistry, so you should inspect the actual output columns before deciding how to define your mantle-cover metric. [web:44]


In [ ]:

params_test = {
    'initialDens': 1e4,
    'initialTemp': 15.0,
    'baseAv': 0.0,
    'radfield': 1.0,
    'zeta': 1.0,
    'finalTime': 1.0e6,
}

physics_df, chemistry_df, rates_df, abundance_df, flag = uclchem.model.cloud(
    param_dict=params_test,
    out_species=['CO','#CO','@CO',
                 'CO2','#CO2','@CO2',
                 'H2O','#H2O','@H2O',
                 'CH4','#CH4','@CH4'],
    return_dataframe=True
)

In [ ]:
chemistry_df

In [ ]:
# Plot the evolution with time of the ice fraction of CO, CO2, H2O, and CH4
plt.figure(figsize=(8, 5))
CO_ice_fraction = (chemistry_df['#CO'] + chemistry_df['@CO']) / (chemistry_df['CO'] + chemistry_df['#CO'] + chemistry_df['@CO'])
plt.plot(physics_df['Time'], CO_ice_fraction, label='CO')
CO2_ice_fraction = (chemistry_df['#CO2'] + chemistry_df['@CO2']) / (chemistry_df['CO2'] + chemistry_df['#CO2'] + chemistry_df['@CO2'])
plt.plot(physics_df['Time'], CO2_ice_fraction, label='CO2')
H2O_ice_fraction = (chemistry_df['@H2O']) / (chemistry_df['H2O'] + chemistry_df['#H2O'] + chemistry_df['@H2O'])
plt.plot(physics_df['Time'], H2O_ice_fraction, label='H2O')
CH4_ice_fraction = (chemistry_df['#CH4'] + chemistry_df['@CH4']) / (chemistry_df['CH4'] + chemistry_df['#CH4'] + chemistry_df['@CH4'])
plt.plot(physics_df['Time'], CH4_ice_fraction, label='CH4')
plt.xscale('log')
plt.xlabel('Time (yr)')
plt.ylabel('Ice Fraction')
plt.title('Ice Fraction Evolution')
plt.legend()



## 3 · Compute the freeze-out timescale for each species

In order to estimate the fraction of grain surface that is covered by ice, we need first to determine the freeze-out timescale. The rate at which molecules freeze out depends on a) how often they hit the grain, determined by gas density, temperature and cross-section of the grain, b) and on whther they stick, determine by the temperature and surface composition.


In [ ]:
def compute_freezeout_timescale(data,species_name,min_time=1):
    # Find the ice species in the dataframe
    ice_prefixes = ['#', '@']
    ice_species = []
    for prefix in ice_prefixes:
        candidate = prefix + species_name
        if candidate in data.columns:
            ice_species.append(candidate)
    
    if len(ice_species) == 0:
        raise ValueError(f"No ice species found for {species_name}")
        return None
    
    # Extract the time evolution
    time = data['Time']
    gas_phase = data[species_name]
    ice_phase = data[ice_species].sum(axis=1)

    # Select the data we care about
    mask = (time >= min_time) & (ice_phase > 0)
    time = time[mask]
    gas_phase = gas_phase[mask]
    ice_phase = ice_phase[mask]

    # Compute the linear fit to the logs
    log_gas = np.log(gas_phase/(ice_phase + gas_phase))
    coeffs = np.polyfit(time,log_gas,1)
    slope = coeffs[0]

    if slope >= 0:
        # No decay (or increasing), return last time
        tau_freeze = time.iloc[-1]
    else:
        tau_freeze = -1.0/slope

    return tau_freeze

In [ ]:
full_df = pd.concat((physics_df, chemistry_df), axis=1)

In [ ]:
CANDIDATE_MOLECULES = ['H2O', 'CO', 'CO2', 'CH4', 'NH3', 'CH3OH', 'H2CO', 'HCN', 'H2S']
MOLECULE_WEIGHTS = [18.015, 28.010, 44.009, 16.043, 17.031, 32.042, 30.026, 27.025, 34.081]  # a.m.u.
freezeout_timescales = {}
for molecule in CANDIDATE_MOLECULES:
    try:
        tau = compute_freezeout_timescale(full_df, molecule)
        freezeout_timescales[molecule] = tau
    except ValueError:
        print(f"Skipping {molecule} as no ice species found")

print("Freeze-out timescales (yr):")
for molecule, tau in freezeout_timescales.items():
    print(f"  {molecule}: {tau:.2e}")

In [ ]:
# Now compute a mass-weighted average freeze-out timescale, using the final abundances of the species as weights
final_abundances = chemistry_df.iloc[-1][CANDIDATE_MOLECULES]
total_abundance = final_abundances.sum()
weighted_tau = sum((final_abundances[molecule] / total_abundance) * freezeout_timescales[molecule] for molecule in CANDIDATE_MOLECULES if molecule in freezeout_timescales)
print(f"Mass-weighted average freeze-out timescale: {weighted_tau:.2e} yr")

# Compute the mass-weighted mean molecular weight of the ice species
mean_molecular_weight = sum((final_abundances[molecule] / total_abundance) * MOLECULE_WEIGHTS[i] for i, molecule in enumerate(CANDIDATE_MOLECULES) if molecule in freezeout_timescales)
print(f"Mass-weighted mean molecular weight of ice species: {mean_molecular_weight:.2f} amu")

# Compute the mass-weighted molecular abundance of the ice species
weighted_abundance = sum((final_abundances[molecule] / total_abundance) * final_abundances[molecule] for molecule in CANDIDATE_MOLECULES if molecule in freezeout_timescales)
print(f"Mass-weighted molecular abundance of ice species: {weighted_abundance:.2e}")

In [ ]:
final_abundances

## 4. Compute the covering fraction with grain size

We first compute the total number of sites assummed inside UCLCHEM:


In [ ]:

SURFACE_SITE_DENSITY = 1.5e15 # cm^-2
GRAIN_RADIUS = 0.1e-4 # cm
GAS_DUST_MASS_RATIO = 100.0
GRAIN_DENSITY = 3.0 # g/cm^3
AMU = 1.66053892e-24 # g
GAS_DUST_DENSITY_RATIO = (4.0 * np.pi * (GRAIN_RADIUS**3) * GRAIN_DENSITY * GAS_DUST_MASS_RATIO)/(3.0 * AMU)
NUM_SITES_PER_GRAIN = 4.0 * np.pi * (GRAIN_RADIUS*GRAIN_RADIUS) * SURFACE_SITE_DENSITY

Based on the total abundances in the surface, this is how UCLCHEM computes the surface coverage:

In [ ]:
surface_abundances = {}
for molecule in CANDIDATE_MOLECULES:
    surface_abundances[molecule] = chemistry_df.iloc[-1][f"#{molecule}"]
all_surface_abundance = sum(surface_abundances[molecule] for molecule in CANDIDATE_MOLECULES if molecule in freezeout_timescales)
surface_coverage = all_surface_abundance * 0.5 * GAS_DUST_DENSITY_RATIO / NUM_SITES_PER_GRAIN
print(f"Surface coverage of grains after {weighted_tau:.2e} yr: {surface_coverage:.2f} monolayers")

In [ ]:
def coverage_fraction_size(radius, grain_density, total_surface_abundance):
    num_sites_per_grain = 4.0 * np.pi * (radius*radius) * SURFACE_SITE_DENSITY
    gas_dust_density_ratio = (4.0 * np.pi * (radius**3) * grain_density * GAS_DUST_MASS_RATIO)/(3.0 * AMU)
    surface_coverage = total_surface_abundance * 0.5 * gas_dust_density_ratio / num_sites_per_grain
    return surface_coverage

In [ ]:
# Plot how the surface coverage changes with grain size for a fixed total surface abundance
grain_sizes = np.logspace(-6, -3, 10) # cm
coverages = [coverage_fraction_size(radius, GRAIN_DENSITY, all_surface_abundance) for radius in grain_sizes]
plt.figure(figsize=(8, 5))
plt.plot(grain_sizes*1e4, coverages, marker='o')
plt.xscale('log')
plt.xlabel('Grain Radius (microns)')
plt.ylabel('Surface Coverage (monolayers)')
plt.title('Surface Coverage vs Grain Size')
plt.grid(True, which='both', ls='--', alpha=0.5)


We use this to compute the covering fraction:

In [ ]:
f_cov = 1.0 - np.exp(-tau_mono_years/weighted_tau)
print(f"Fractional coverage of grains after {weighted_tau:.2e} yr: {f_cov:.2f}")
print(tau_mono_years, weighted_tau, f_cov,np.exp(-weighted_tau / tau_mono_years))


## 5 · Parameter grid

A first-pass grid can span \(n_{
m H}\), temperature, UV field, and grain size. Because each UCLCHEM model is a separate single-point solve, the full sweep is naturally suitable for process-based parallelism in Python. [web:97]


In [ ]:

nH_grid = np.logspace(2, 6, 5)
Tgas_grid = np.array([10, 12, 15, 18, 20, 25, 30, 40, 60, 100], dtype=float)
G0_grid = np.array([0.01, 0.1, 1.0, 10.0, 100.0], dtype=float)
grain_size_um_grid = np.array([0.01, 0.03, 0.1, 0.3, 1.0], dtype=float)

param_grid = list(product(nH_grid, Tgas_grid, G0_grid, grain_size_um_grid))
print('Total jobs:', len(param_grid))



## 6 · Worker-safe UCLCHEM setup

Each worker should write to its own temporary file and return a plain dictionary. That keeps the parallel launch robust even if the wrapped code uses internal file I/O.


In [ ]:

def make_uclchem_params(nH, Tgas, G0, grain_size_um, output_file):
    return {
        'initialDens': float(nH),
        'finalDens': float(nH),
        'initialTemp': float(Tgas),
        'finalTemp': float(Tgas),
        'rout': 0.05,
        'rin': 0.0,
        'baseAv': 5.0,
        'radfield': float(G0),
        'zeta': 1.3e-17,
        'phase': 1,
        'collapse': 0,
        'freezeout': 1,
        'desorb': 1,
        'thermdesorb': 1,
        'uvdesorb': 1,
        'crdesorb': 1,
        'endTime': 1.0e7,
        'outputTime': 1.0e7,
        'grain_rad': float(grain_size_um) * 1e-4,
        'network': 'networks/gas_grain_full.dat',
        'outputFile': str(output_file),
    }


def run_one_model(job):
    nH, Tgas, G0, grain_size_um = job
    import tempfile
    from pathlib import Path
    # import uclchem

    with tempfile.TemporaryDirectory(prefix='uclchem_job_') as tmpdir:
        outfile = Path(tmpdir) / 'uclchem_output.dat'
        params = make_uclchem_params(nH, Tgas, G0, grain_size_um, outfile)
        try:
            result = uclchem.model.cloud(params)
        except Exception as e:
            return {
                'nH': nH,
                'Tgas': Tgas,
                'G0': G0,
                'grain_um': grain_size_um,
                'status': 'failed',
                'error': repr(e),
            }
        if result is None or len(result) == 0:
            return {
                'nH': nH,
                'Tgas': Tgas,
                'G0': G0,
                'grain_um': grain_size_um,
                'status': 'failed',
                'error': 'empty result',
            }
        species_pairs = find_existing_ice_species(result.columns)
        last = result.iloc[-1]
        covering, per_species_frac = build_covering_metric(last, species_pairs, weights=None)
        per_species_ice = compute_species_ice_abundance(last, species_pairs)
        row = {
            'nH': nH,
            'Tgas': Tgas,
            'G0': G0,
            'grain_um': grain_size_um,
            'status': 'ok',
            'n_species_used': len(species_pairs),
            'covering_metric': covering,
        }
        for gas_name, _ in species_pairs:
            row[f'freezeout_{gas_name}'] = per_species_frac.get(gas_name, 0.0)
            row[f'iceabund_{gas_name}'] = per_species_ice.get(gas_name, 0.0)
        return row



## 7 · Parallel launch

On macOS and inside notebooks, the `spawn` context is usually the safest multiprocessing option. The Python standard library documents `Pool`-based process parallelism for distributing independent jobs across CPUs. [web:97]


In [ ]:

def run_grid_parallel(param_grid, nproc=None, chunksize=1):
    if nproc is None:
        nproc = max(1, min(cpu_count() - 1, 4))
    ctx = get_context('spawn')
    with ctx.Pool(processes=nproc) as pool:
        results = list(pool.imap_unordered(run_one_model, param_grid, chunksize=chunksize))
    return pd.DataFrame(results)

# df_all = run_grid_parallel(param_grid, nproc=4, chunksize=2)
# df_all.head()



## 8 · Inspect failures before science plots

Large sweeps often fail first because of environment or naming issues, not chemistry. Always separate failed and successful jobs before moving on.


In [ ]:

# print(df_all['status'].value_counts(dropna=False))
# failed = df_all[df_all['status'] != 'ok']
# good = df_all[df_all['status'] == 'ok'].copy()
# display(failed.head())



## 9 · Determine which species dominate the mantle in your grid

Compute each species’ mean fractional contribution to the total ice abundance and keep only those above a threshold, for example 5 percent. That turns the candidate set into a data-driven dominant set.


In [ ]:

def summarize_dominant_ices(df_good, min_mean_fraction=0.05):
    ice_cols = [c for c in df_good.columns if c.startswith('iceabund_')]
    tmp = df_good[ice_cols].copy().fillna(0.0)
    total = tmp.sum(axis=1)
    frac = tmp.div(total.replace(0, np.nan), axis=0).fillna(0.0)
    mean_frac = frac.mean(axis=0).sort_values(ascending=False)
    dominant = mean_frac[mean_frac >= min_mean_fraction]
    return mean_frac, dominant

# mean_frac, dominant = summarize_dominant_ices(good, min_mean_fraction=0.05)
# display(mean_frac.to_frame('mean_fraction'))
# print(list(dominant.index))


In [ ]:

# ax = mean_frac.sort_values().plot(kind='barh', figsize=(7, 4))
# ax.set_xlabel('Mean fractional contribution to total ice abundance')
# ax.set_title('Dominant mantle species across the sampled grid')
# plt.tight_layout()



## 10 · Map covering to accretion efficiency

Once you have a covering metric, choose a phenomenological mapping to \(\epsilon_{
m acc}\). For your science question, a non-linear mapping is often better than a linear one because it distinguishes between “a little ice exists” and “the original grain surface is effectively buried.”


In [ ]:

def epsilon_linear(C, eta=0.9):
    return np.clip(1.0 - eta * np.asarray(C), 0.0, 1.0)


def epsilon_sigmoid(C, C0=0.4, alpha=12.0):
    C = np.asarray(C)
    return 1.0 / (1.0 + np.exp(alpha * (C - C0)))


def epsilon_powerlaw(C, p=2.0):
    C = np.asarray(C)
    return np.clip((1.0 - C) ** p, 0.0, 1.0)

# good['eps_linear'] = epsilon_linear(good['covering_metric'])
# good['eps_sigmoid'] = epsilon_sigmoid(good['covering_metric'])
# good['eps_powerlaw'] = epsilon_powerlaw(good['covering_metric'])



## 11 · Visualize transition surfaces

Make \((n_{
m H}, T)\) slices at fixed \(G_0\) and grain size for both the covering metric and your chosen \(\epsilon_{
m acc}\) mapping.


In [ ]:

def make_2d_slice(df, value_col, fixed_G0=1.0, fixed_grain=0.1):
    sub = df[(df['status'] == 'ok') & (np.isclose(df['G0'], fixed_G0)) & (np.isclose(df['grain_um'], fixed_grain))].copy()
    nH_u = np.sort(sub['nH'].unique())
    T_u = np.sort(sub['Tgas'].unique())
    Z = np.full((len(T_u), len(nH_u)), np.nan)
    for i, T in enumerate(T_u):
        for j, nH in enumerate(nH_u):
            m = sub[(np.isclose(sub['Tgas'], T)) & (np.isclose(sub['nH'], nH))]
            if len(m) > 0:
                Z[i, j] = m.iloc[0][value_col]
    return nH_u, T_u, Z

# nH_u, T_u, Z = make_2d_slice(good.assign(eps=epsilon_powerlaw(good['covering_metric'], p=2)), 'eps')
# fig, ax = plt.subplots(figsize=(7, 5))
# cs = ax.contourf(np.log10(nH_u), T_u, Z, levels=np.linspace(0, 1, 21), cmap='plasma')
# fig.colorbar(cs, ax=ax, label='epsilon_acc')
# ax.set_xlabel('log10(nH / cm^-3)')
# ax.set_ylabel('Tgas [K]')
# plt.tight_layout()



## 12 · Fit a compact reduced model

You can fit the covering metric directly instead of fitting each species separately. That is usually the right reduction step for RAMSES.


In [ ]:

def compact_covering_model(X, T0, dlogn, betaG, betaa, alpha):
    lognH, Tgas, G0, loga = X
    Tcrit = T0 + dlogn * (lognH - 3.0)
    Teff = Tgas + betaG * G0 + betaa * loga
    C = 1.0 / (1.0 + np.exp(alpha * (Teff / Tcrit - 1.0)))
    return np.clip(C, 0.0, 1.0)

# X = (
#     np.log10(good['nH'].values),
#     good['Tgas'].values,
#     good['G0'].values,
#     np.log10(good['grain_um'].values / 0.1),
# )
# y = good['covering_metric'].values
# popt, pcov = curve_fit(compact_covering_model, X, y, p0=[20, 3, 2, -1, 8], maxfev=10000)
# print(popt)



## 13 · Export outputs

Export the grid table, the dominant-species summary, and optionally the fitted parameters. That gives you both a transparent data product and a compact prescription for RAMSES.


In [ ]:

# good.to_csv('uclchem_multiice_parallel_grid.csv', index=False)
# dominant.to_frame('mean_fraction').to_csv('uclchem_dominant_ices.csv')
# np.savetxt('uclchem_compact_covering_fit.txt', np.array(popt))



## 14 · Jupyter and multiprocessing notes

If multiprocessing inside the notebook is flaky on your machine, keep the exploratory cells in the notebook and move only the worker and pool launcher into a tiny helper script. That still preserves the same workflow while avoiding notebook-specific process-launch quirks. The standard library multiprocessing model is still the right basic approach for this kind of embarrassingly parallel grid. [web:97]
